# Training the model

## Importing Dependencies

In [43]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV,KFold,cross_val_score
from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score,root_mean_squared_error
import warnings
warnings.filterwarnings('ignore')
import pickle

## Data Ingestion

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/refs/heads/master/titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Checking for Missing Values

In [3]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

## Insight
- There are missing values in 3 columns
- Age column is quantitative feature so we will use mean
- Cabin is qualitative feature and more than 80% null values so we will drop it.
- Embarked column is qualitative feature so we will use mode

## Imputation

In [4]:
df['Age'].fillna(df["Age"].mean(),inplace=True)
df.drop(['Cabin','PassengerId'],axis=1,inplace=True) 
df['Embarked'].fillna(df['Embarked'].mode()[0],inplace=True)

Note - 'PassengerId' have only numbers from 1 to 891

## Checking for duplicates

In [5]:
df.duplicated().sum()

np.int64(0)

## Insight
There are no duplicate values

## Separating Dependant and independant Feature


In [6]:
X = df.drop(['Fare','Name'],axis=1)
y = df[['Fare']]

In [8]:
##Label encoder object was not able to fit testing set
lbl_ticket = LabelEncoder()
X['Ticket'] = lbl_ticket.fit_transform(X['Ticket'])


## Holdout Validation (Train_test_split)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [10]:
X_train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Embarked
298,1,1,male,29.699118,0,0,98,S
884,0,3,male,25.000000,0,0,650,S
247,1,2,female,24.000000,0,2,166,S
478,0,3,male,22.000000,0,0,411,S
305,1,1,male,0.920000,1,2,37,S


## Encoding

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Name      891 non-null    object 
 3   Sex       891 non-null    object 
 4   Age       891 non-null    float64
 5   SibSp     891 non-null    int64  
 6   Parch     891 non-null    int64  
 7   Ticket    891 non-null    object 
 8   Fare      891 non-null    float64
 9   Embarked  891 non-null    object 
dtypes: float64(2), int64(4), object(4)
memory usage: 69.7+ KB


## Insight 
- There are 4 object based features
- Name will be drop, as it have personal info
- For Sex and Embarked features, we will use OneHotEncoding as this is nominal data
- For ticket we will use label encoding as this have too much categories

In [12]:
## Initializing/Loading Object
ohe_sex = OneHotEncoder(sparse_output=False)
ohe_embark = OneHotEncoder(sparse_output=False)


In [13]:
## One hot encoding
X_train_Sex = ohe_sex.fit_transform(X_train[['Sex']])
X_test_Sex = ohe_sex.transform(X_test[['Sex']])
X_train_embark = ohe_embark.fit_transform(X_train[['Embarked']])
X_test_embark = ohe_embark.transform(X_test[['Embarked']])


In [14]:
## Converting to dataframe 

In [15]:
X_train_Sex = pd.DataFrame(X_train_Sex,columns=ohe_sex.get_feature_names_out()).reset_index(drop=True)
X_test_Sex = pd.DataFrame(X_test_Sex,columns=ohe_sex.get_feature_names_out()).reset_index(drop=True)
X_train_embark = pd.DataFrame(X_train_embark,columns=ohe_embark.get_feature_names_out()).reset_index(drop=True)
X_test_embark = pd.DataFrame(X_test_embark,columns=ohe_embark.get_feature_names_out()).reset_index(drop=True)

In [16]:
## Final check
X_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 223 entries, 709 to 424
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  223 non-null    int64  
 1   Pclass    223 non-null    int64  
 2   Sex       223 non-null    object 
 3   Age       223 non-null    float64
 4   SibSp     223 non-null    int64  
 5   Parch     223 non-null    int64  
 6   Ticket    223 non-null    int64  
 7   Embarked  223 non-null    object 
dtypes: float64(1), int64(5), object(2)
memory usage: 15.7+ KB


## Concat on dataset

In [17]:
Xtrain_final = pd.concat([X_train.reset_index(drop=True),X_train_Sex,X_train_embark],axis=1)
Xtest_final = pd.concat([X_test.reset_index(drop=True),X_test_Sex,X_test_embark],axis=1)
Xtrain_final.drop(['Sex','Embarked'],axis=1,inplace=True)
Xtest_final.drop(['Sex','Embarked'],axis=1,inplace=True)

In [18]:
Xtrain_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 668 entries, 0 to 667
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Survived    668 non-null    int64  
 1   Pclass      668 non-null    int64  
 2   Age         668 non-null    float64
 3   SibSp       668 non-null    int64  
 4   Parch       668 non-null    int64  
 5   Ticket      668 non-null    int64  
 6   Sex_female  668 non-null    float64
 7   Sex_male    668 non-null    float64
 8   Embarked_C  668 non-null    float64
 9   Embarked_Q  668 non-null    float64
 10  Embarked_S  668 non-null    float64
dtypes: float64(6), int64(5)
memory usage: 57.5 KB


## Scaling

In [19]:
scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(Xtrain_final)
X_test_scaled = scalar.transform(Xtest_final)

## Modeling

In [20]:
reg_rf = RandomForestRegressor()
reg_ab = AdaBoostRegressor()

In [21]:
## Training a baseline model
def evaluator(predicted):
  print(f'MSE - {mean_squared_error(y_test,predicted)}')
  print(f'MAE - {mean_absolute_error(y_test,predicted)}')
  print(f'R2 Score - {r2_score(y_test,predicted)}')
  print(f'RMSE - {root_mean_squared_error(y_test,predicted)}')
def trainer(object):
  object.fit(X_train_scaled,y_train)
  y_pred = object.predict(X_test_scaled)
  return evaluator(y_pred)

In [22]:
trainer(reg_ab)

MSE - 1691.3823691715252
MAE - 17.445461633167763
R2 Score - 0.10535073867635847
RMSE - 41.1264193575313


In [23]:
trainer(reg_rf)

MSE - 1428.3112194801024
MAE - 12.764977542600894
R2 Score - 0.24450106567324859
RMSE - 37.79300490143781


## Cross Validation

In [24]:
cv = KFold(n_splits=5,shuffle=True,random_state=42)

In [25]:
model = cross_val_score(estimator=reg_rf,X=X_train_scaled,y=y_train,scoring='r2',cv=cv)

In [26]:
model.mean()

np.float64(0.7183990782111029)

## HyperParameter Tuning

In [30]:
param_grid = {
    'n_estimators' : [100,150,200,250],
    'criterion' : ['squared_error', "absolute_error", "poisson"],
    'max_depth' : [3,4,5,6],
    'min_samples_split' : [2,3,4],
    'oob_score' : [True,False]
}

In [33]:
reg = RandomForestRegressor()
grid  = GridSearchCV(estimator=reg,param_grid=param_grid,scoring='neg_root_mean_squared_error' ,cv=5,verbose=3)

In [34]:
grid.fit(X_train_scaled,y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV 1/5] END criterion=squared_error, n_estimators=100;, score=-15.125 total time=   0.5s
[CV 2/5] END criterion=squared_error, n_estimators=100;, score=-21.659 total time=   0.4s
[CV 3/5] END criterion=squared_error, n_estimators=100;, score=-38.671 total time=   0.5s
[CV 4/5] END criterion=squared_error, n_estimators=100;, score=-22.526 total time=   0.6s
[CV 5/5] END criterion=squared_error, n_estimators=100;, score=-27.111 total time=   0.5s
[CV 1/5] END criterion=squared_error, n_estimators=150;, score=-15.207 total time=   0.8s
[CV 2/5] END criterion=squared_error, n_estimators=150;, score=-21.773 total time=   1.0s
[CV 3/5] END criterion=squared_error, n_estimators=150;, score=-40.295 total time=   0.6s
[CV 4/5] END criterion=squared_error, n_estimators=150;, score=-22.515 total time=   0.9s
[CV 5/5] END criterion=squared_error, n_estimators=150;, score=-29.886 total time=   0.6s
[CV 1/5] END criterion=squared_error, n

,estimator,RandomForestRegressor()
,param_grid,"{'criterion': ['squared_error', 'absolute_error', ...], 'n_estimators': [100, 150, ...]}"
,scoring,'neg_root_mean_squared_error'
,n_jobs,None
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [35]:
grid.best_params_

{'criterion': 'squared_error', 'n_estimators': 100}

In [36]:
{'criterion': 'squared_error',
 'max_depth': 6,
 'min_samples_split': 2,
 'n_estimators': 100,
 'oob_score': True}

{'criterion': 'squared_error',
 'max_depth': 6,
 'min_samples_split': 2,
 'n_estimators': 100,
 'oob_score': True}

In [37]:
reg_final = RandomForestRegressor(criterion='squared_error',max_depth=6,min_samples_split=2,n_estimators=100,oob_score=True)

In [38]:
trainer(reg_final)

MSE - 1365.774733236333
MAE - 12.946400941879903
R2 Score - 0.27757946488298435
RMSE - 36.956389613114716


In [39]:
reg_final

,n_estimators,100
,criterion,'squared_error'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,True


# Inferencing

### Pickling

In [44]:
## saving ML model
with open('model.pkl','wb') as a:
    pickle.dump(reg_final,a)

In [45]:
## Saving Encoder objects
with open('ohe_sex.pkl','wb') as a1:
    pickle.dump(ohe_sex,a1)
with open('ohe_embark.pkl','wb') as a2:
    pickle.dump(ohe_embark,a2)
with open('lbl_ticket.pkl','wb') as a3:
    pickle.dump(lbl_ticket,a3)

In [46]:
## Saving Scaler object
with open('scalar.pkl','wb') as a4:
    pickle.dump(scalar,a4)

In [49]:
X_train_scaled[0].size

11

In [54]:
Xtrain_final.sample()

,Survived,Pclass,Age,SibSp,Parch,Ticket,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
130,0,2,16.0,0,0,619,0.0,1.0,0.0,0.0,1.0


In [51]:
test_data = np.array([0,2,18,0,0,22,0,1,0,1,0])

In [53]:
test_data

array([ 0,  2, 18,  0, 22,  0,  1,  0,  1,  0])

In [58]:
reg_final.predict([X_train_scaled[77]])

array([17.89380446])

In [65]:
y_train.iloc[77]

Fare    20.525
Name: 328, dtype: float64

In [69]:
print(reg_final.predict([X_train_scaled[127]]))
print(y_train.iloc[127])

[66.28518889]
Fare    57.9792
Name: 523, dtype: float64
